In [9]:
import numpy as np
from sklearn.datasets import load_iris, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.datasets import load_digits

In [10]:
kernels = ['linear', 'poly', 'rbf']

iris = load_iris()
x = iris.data
y = iris.target
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

x_train, x_test, y_train, y_test = train_test_split(
    x_scaled, y, test_size=0.2, random_state=42
)

In [11]:
iris_results = {}
for k in kernels:
    model = SVC(kernel=k, random_state=42)
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    acc = accuracy_score(y_test, pred)
    iris_results[k] = acc
    print(k, "->", acc)

best_kernel = max(iris_results, key=iris_results.get)
model_best = SVC(kernel=best_kernel, random_state=42)
model_best.fit(x_train, y_train)
pred_best = model_best.predict(x_test)

print("best kernel on iris was", best_kernel)
print(classification_report(y_test, pred_best, target_names=iris.target_names))

linear -> 0.9666666666666667
poly -> 0.9666666666666667
rbf -> 1.0
best kernel on iris was rbf
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [12]:
# MNIST part
mnist = load_digits()

x_mnist = mnist.data
y_mnist = mnist.target

x_mnist = x_mnist[:10000]
y_mnist = y_mnist[:10000]

scaler_m = StandardScaler()
x_mnist_scaled = scaler_m.fit_transform(x_mnist)

x_train_m, x_test_m, y_train_m, y_test_m = train_test_split(
    x_mnist_scaled, y_mnist, test_size=0.2, random_state=42
)

print("mnist subset train", x_train_m.shape, "test", x_test_m.shape)

mnist_results = {}

mnist subset train (1437, 64) test (360, 64)


In [13]:
ova_linear = OneVsRestClassifier(SVC(kernel='linear', random_state=42))
ova_linear.fit(x_train_m, y_train_m)

pred_linear = ova_linear.predict(x_test_m)
mnist_results['linear'] = accuracy_score(y_test_m, pred_linear)
print("linear kernel done, acc", mnist_results['linear'])

ova_poly = OneVsRestClassifier(SVC(kernel='poly', degree=3, random_state=42))
ova_poly.fit(x_train_m, y_train_m)

pred_poly = ova_poly.predict(x_test_m)
mnist_results['poly'] = accuracy_score(y_test_m, pred_poly)
print("poly kernel done, acc", mnist_results['poly'])

ova_rbf = OneVsRestClassifier(SVC(kernel='rbf', random_state=42))
ova_rbf.fit(x_train_m, y_train_m)

pred_rbf = ova_rbf.predict(x_test_m)
mnist_results['rbf'] = accuracy_score(y_test_m, pred_rbf)
print("rbf kernel done, acc", mnist_results['rbf'])

preds_by_kernel = {
    'linear': pred_linear,
    'poly': pred_poly,
    'rbf': pred_rbf
}

linear kernel done, acc 0.9527777777777777
poly kernel done, acc 0.9777777777777777
rbf kernel done, acc 0.9833333333333333


In [14]:
print("kernel comparison")
for k in kernels:
    print(k, "iris:", iris_results[k], " mnist:", mnist_results[k])

best_mnist = max(mnist_results, key=mnist_results.get)

print("best kernel on mnist was", best_mnist)
print(classification_report(y_test_m, preds_by_kernel[best_mnist]))

kernel comparison
linear iris: 0.9666666666666667  mnist: 0.9527777777777777
poly iris: 0.9666666666666667  mnist: 0.9777777777777777
rbf iris: 1.0  mnist: 0.9833333333333333
best kernel on mnist was rbf
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        33
           1       1.00      1.00      1.00        28
           2       0.97      1.00      0.99        33
           3       1.00      0.97      0.99        34
           4       1.00      1.00      1.00        46
           5       0.96      0.98      0.97        47
           6       0.97      1.00      0.99        35
           7       0.97      0.97      0.97        34
           8       1.00      0.97      0.98        30
           9       0.97      0.95      0.96        40

    accuracy                           0.98       360
   macro avg       0.98      0.98      0.98       360
weighted avg       0.98      0.98      0.98       360

